# YZ50 — Week 7: Building Self-Attention

- **Character-level language modeling** turns raw text into a sequence of tokens that can be used to train a model to predict the **next character** from a given context.

- In this week, the **Tiny Shakespeare dataset** replaces the previous names dataset, and a **character-level tokenizer** is built using `encode` and `decode` functions. The dataset is then divided into **training and validation sets**, while random context windows are sampled to construct mini-batches.

- A simple **Bigram language model** is first trained as a **baseline**, providing a reference validation loss before introducing attention and allowing the effect of the new architecture to be measured directly.

- The idea behind attention is introduced by computing the **average of previous token representations** in multiple equivalent ways: using explicit loops, **lower-triangular matrix multiplication**, and **softmax-based weighted aggregation**. This demonstrates how matrix multiplication can efficiently combine information from previous positions.

- **Self-attention** replaces uniform averaging with **learned, data-dependent attention weights**. Each token produces a **query, key, and value**, allowing it to determine which previous tokens are most relevant to its current representation.

- A **causal lower-triangular mask** prevents tokens from accessing future positions, ensuring that predictions only depend on information that would actually be available during autoregressive text generation.

- Because self-attention itself does not encode token order, **positional embeddings** are added to token embeddings so that the model can distinguish where each character appears within the context.

- Attention scores are scaled by **\(1/\sqrt{\text{head\_size}}\)** before applying softmax, preventing the dot products from becoming too large and producing excessively sharp attention distributions.

In this week, I move from **fixed-context character prediction** toward **context-dependent information aggregation**, build a **single self-attention head from scratch**, integrate it into a character-level language model, and explore how **queries, keys, values, causal masking, positional information, and scaled attention** form the core mechanism behind Transformer-based language models.

# 1. Veriyi hazırla ve tabanı kur. Tiny Shakespeare'i oku, karakter tokenizer'ını yaz (encode, decode), veriyi %90 train, %10 val diye ayır. Rastgele parçalardan batch çeken get_batch fonksiyonunu yaz. Hafta 2'deki bigram modelini nn.Module olarak kur, eğit, val loss'u kaydet. Bu senin karşılaştırma tabanın.

## 1.1 Read Tiny Shakespeare

In [1]:
# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [2]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


In [3]:
# let's look at the first 1000 characters
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



## 1.2 Write the character tokenizer (encode, decode)

In [4]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text))) # possible elements of the vocabulary
vocab_size = len(chars) # the number of unique characters
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


**Tokenization** is the process of converting raw text into smaller units called **tokens** and mapping these tokens to numerical representations that a language model can process.

A **tokenizer** converts text into numerical tokens using **encoding** and converts those tokens back into text using **decoding**.

- **Encode:** text → token IDs
- **Decode:** token IDs → text

In [5]:
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }

# Tokeization functions
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


### Subword Tokenization

A **subword tokenizer** splits text into units that are smaller than full words but can contain multiple characters. This provides a balance between character-level and word-level tokenization.

For example:

`"unbelievable"` → `"un"` + `"believ"` + `"able"`

**BPE (Byte Pair Encoding)** is a subword tokenization algorithm that starts with small units and repeatedly merges frequently occurring pairs into larger tokens. Common sequences can therefore become single tokens, while rare words can still be represented using smaller pieces.

**GPT-2** uses a **byte-level BPE tokenizer**, where text is ultimately represented using byte-based subword tokens. This allows the tokenizer to represent essentially any input text without requiring a special unknown-word token.

**tiktoken** is OpenAI's fast **BPE-based tokenizer library** used with OpenAI models. It provides the encoding and decoding implementations for tokenization schemes used by models in the GPT family.

In short:

**Text → Tokenizer (BPE) → Token IDs → Model**

and

**Token IDs → Tokenizer → Text**

In [6]:
# let's now encode the entire text dataset and store it into a torch.Tensor
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000]) # the 1000 characters we looked at earier will to the GPT look like this

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

## 1.3 Split the data into 90% training and 10% validation sets

In [7]:
# Let's now split up the data into train and validation sets
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [8]:
block_size = 8 # context length for predictions
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

* there is a 8 different information in this, for example 47 comes after [18], 56 comes after [18, 47], 57 comes after [18, 47, 56], 58 comes after [18, 47, 56, 57], 1 comes after [18, 47, 56, 57, 58], and so on. 

In [9]:
x = train_data[:block_size]
y = train_data[1:block_size+1] # targets for each position in the input sequence, shifted by one character
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


## 1.4 Write a get_batch function that retrieves batches from random batches

In [10]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")


# 32 independet examples, packed into a single batch, each of length 8. For each of the 32 examples, we have 8 different predictions to make, one for each character in the context.

# 4x8 tensor, 4 batches, 8 characters in each batch. Each character is represented by an integer, which is the index of that character in the vocabulary.

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53

In [11]:
print(xb) # our input to the transformer

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


## 1.5 Implement the bigram model as an nn.Module, train it, and record the validation loss

In [12]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C), where C = vocab_size

        if targets is None:
            loss = None
        else:
            # reshape the logits and targets to be 2D tensors for the loss function
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [13]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [14]:
batch_size = 32
for steps in range(1000): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())


3.704136848449707


In [15]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))


Wh;;Sq.f ustNzknc
kwgOj$dhPWr,SV?hsusiKpgXXUh;Apmem d?hESXI.i;TrJgkiF-oKbXCAA -botrngFCHAUQkn$

pn$w-gHoi?wtd!
LLULIfSK'bAw :M.ZtOptXEQcL?hfaofqbPd?OnonQQJMap$aypupIBYGUsZaI'ottllo..k$W$Akp?yl?ajKlzY!lx&QQLW? t,bXFkyhl-dmVsHeckhRl,jSClgjuk:3Iv
?OqlrV;!Plxfzgy;;
'mRjuBQ&xk!$
h
SiruDJgKuDny,S$ERf.?GSV-ivvKcOvi-nQGX&q-YQbm dEM?px;Akr-IESq--wIWId
RFgXTpDUgM:CK$I!uo'IBT -
j?wfy fFr.&fiqtRS.ZttxGh' a!ogrn$zoZqbocL&yIffBDWNUboscuQqo.Fls,?,M?eZxHx?p?EV.mJiHqHnxT  bQpa;P fawiF$-QbWv&f:CVDCBfano,b?$Esev.?


# 2. Geçmişin ortalamasını üç yoldan al: for döngüsüyle, lower triangular matrisle (torch.tril) çarparak, softmax'la. Üçünün aynı sonucu verdiğini torch.allclose ile göster. Matris çarpımının neden ağırlıklı ortalama olduğunu kendi cümlelerinle yaz. 

## The mathematical trick in self-attention

### Token Communication and Context Aggregation

Suppose we have **eight tokens** in a sequence. Initially, these tokens do not communicate with each other, but we would like them to **exchange information** in a very specific way.

For example, the token at the **5th position** should not communicate with the tokens at positions **6, 7, and 8**, because these are **future tokens** in the sequence. Instead, it should only receive information from the tokens at positions **1, 2, 3, and 4**, together with its own information.

In other words, information should only flow from the **previous context toward the current timestep**. We cannot use information from future tokens because the goal of the language model is precisely to **predict what comes next**.

The simplest way to allow tokens to communicate is to **average the representations of the preceding tokens**. For example, for the token at position 5, we can average the feature vectors of the tokens at positions **1 through 5**.

The resulting vector then represents an aggregated summary of:

> **the current token + its previous context**

This simple averaging operation introduces the fundamental idea that a token's representation can incorporate information from its **context history**, which will later be extended into a learned weighted aggregation through **self-attention**.

In [16]:
# consider the following toy example:

torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [17]:
# We want x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B,T,C)) # bag of words representation
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # (t,C)
        xbow[b,t] = torch.mean(xprev, 0)

In [18]:
x[0]

tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])

In [19]:
xbow[0]

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

In [20]:
# toy example illustrating how matrix multiplication can be used for a "weighted aggregation"
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [21]:
# version 2: using matrix multiply for a weighted aggregation
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)


xbow2 = wei @ x # (B, T, T) @ (B, T, C) ----> (B, T, C)
torch.allclose(xbow, xbow2, atol=1e-6) # use atol=1e-6 to avoid floating point issues

True

In [22]:
wei

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

### Weighted Aggregation with Matrix Multiplication

`torch.tril` creates a **lower-triangular matrix**, ensuring that each token can only access itself and the tokens before it.

Each row is then **normalized to sum to 1**, so the entries become averaging weights.

Multiplying `wei @ x` computes the **weighted average of all previous token representations** for every timestep simultaneously.

Therefore:

**lower-triangular mask → normalize weights → weighted aggregation of previous context**


### Why Does Matrix Multiplication Compute a Weighted Average?

Matrix multiplication works as a **weighted average** because each output row is formed by multiplying the previous token representations by the corresponding values in `wei` and then adding them together.

Since each row of `wei` **sums to 1**, these values act as weights. The lower-triangular structure also makes the weights of future tokens zero, so each token only combines information from **itself and its previous context**.

For example, if a row of `wei` is:

`[0.25, 0.25, 0.25, 0.25, 0, 0, 0, 0]`

then matrix multiplication takes **25% of each of the first four token representations** and adds them together. Therefore, `wei @ x` computes a weighted average of the available previous tokens.

In [23]:
# version 3: use Softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3, atol=1e-6)

# I didn't quite grasp the operating logic of this version.

True

# 3. Tek bir self-attention head yaz: query, key, value ve pozisyon embedding'i. Bir örnekte ağırlık matrisini (wei) yazdır, bir satırını oku: o token hangi token'lara ne kadar bakıyor. Skorları head_size'ın kareköküne neden böldüğümüzü sayıyla göster: bölmeden ve bölerek softmax çıktısını yan yana koy.

### The Core Idea of Self-Attention

A simple average treats all previous tokens as **equally important**. However, different tokens may find different parts of their context more or less relevant, and we want this interaction to be **data-dependent**.

For example, if the current token is a **vowel**, it might be useful for it to look for certain **consonants in its previous context**. Instead of blindly averaging everything that came before, the token should be able to decide **which previous tokens contain useful information** and gather more information from those tokens.

This is the problem that **self-attention** solves.

Each token produces three different vectors:

- **Query (Q):** *What am I looking for?*
- **Key (K):** *What information do I contain / what can I match with?*
- **Value (V):** *What information will I communicate if another token finds me relevant?*

The **query** of one token is compared with the **keys** of the other tokens using a dot product:

`attention score = query · key`

If a query and a key are strongly aligned, their dot product becomes larger. This means that the corresponding token is considered more relevant to the token making the query.

Therefore, instead of using fixed and uniform weights such as:

`[0.25, 0.25, 0.25, 0, ...]`

self-attention produces **data-dependent attention weights**. Different inputs can therefore produce different attention patterns.

Before normalization, these query-key dot products are raw **affinity scores**. Since this is an autoregressive language model, scores corresponding to **future tokens are masked out** so that a token cannot obtain information from positions that have not occurred yet.

The remaining scores are passed through **softmax**, converting them into positive weights that sum to 1. These weights describe how much information the current token should gather from each available previous token.

Finally, the model does not directly aggregate the original token representations. Each token also produces a **value vector**, and the attention weights are used to compute a weighted combination of these values:

`output = attention_weights @ values`

Therefore, the roles of the three vectors can be summarized as:

**Query → what I am looking for**  
**Key → what I can be matched by**  
**Value → what I will communicate**

Self-attention can therefore be viewed as a **data-dependent communication mechanism between tokens**: queries and keys determine **who should communicate with whom**, while values determine **what information is actually communicated**.

In [24]:
# Self-attention

torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# let's see a single Head perform self-attention
head_size = 16 # hyperparameter: the number of channels in the head
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)   # (B, T, 16)
q = query(x) # (B, T, 16)
wei =  q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ---> (B, T, T)

tril = torch.tril(torch.ones(T, T))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v
#out = wei @ x

out.shape

torch.Size([4, 8, 16])

In [25]:
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)

### Additional Notes on Attention

#### Positional Information

Self-attention itself has **no inherent notion of token position or order**. It simply operates on a set of vectors and determines how strongly they should interact.

Therefore, we explicitly add **positional embeddings** to token embeddings:

`x = token_embedding + position_embedding`

This allows each token representation to contain information about both **what the token is** and **where it appears in the sequence**.

---

#### Independent Batches

Examples across the **batch dimension are processed completely independently** and never communicate with each other.

For an input with shape:

`(B, T, C)`

each of the `B` examples gets its own independent `T × T` attention matrix.

Therefore:

`wei.shape = (B, T, T)`

Communication happens between tokens **within the same sequence**, never between different sequences in the batch.

---

#### Encoder vs. Decoder Attention

The triangular causal mask determines whether tokens are allowed to communicate with future positions.

In **causal / decoder-style attention**, the lower-triangular mask prevents tokens from attending to future tokens. This is necessary for **autoregressive language modeling**, where the model predicts future tokens from previous ones.

If the triangular mask is removed, tokens can communicate with **both previous and future tokens**. This corresponds to **bidirectional attention**, commonly associated with encoder-style Transformer blocks.

---

#### Self-Attention vs. Cross-Attention

In **self-attention**, queries, keys, and values are all produced from the **same input**:

`x → Q, K, V`

In **cross-attention**, the queries are produced from one source, while the keys and values come from a **different external source**:

`x → Q`

`external source → K, V`

Therefore, self-attention allows tokens within the same sequence to communicate, while cross-attention allows one representation to retrieve information from another representation.

---

#### Scaled Dot-Product Attention

The raw attention scores are computed as:

`Q @ Kᵀ`

As `head_size` increases, these dot products tend to grow in magnitude. Large attention logits can make the **softmax distribution extremely sharp**, causing it to concentrate almost all probability on only one or a few tokens.

To prevent this, the attention scores are scaled by:

`1 / sqrt(head_size)`

before applying softmax:

`wei = (Q @ Kᵀ) / sqrt(head_size)`

When the components of `Q` and `K` have approximately unit variance, this scaling helps keep the variance of the attention scores under control.

As a result:

**Without scaling → larger logits → sharper softmax → easier saturation**

**With scaling → controlled logits → more diffuse and stable softmax**

In [26]:
k = torch.randn(B,T,head_size)
q = torch.randn(B,T,head_size)
wei = q @ k.transpose(-2, -1) * head_size**-0.5

In [27]:
k.var()

tensor(1.0449)

In [28]:
q.var()

tensor(1.0700)

In [29]:
wei.var()

tensor(1.0918)

In [30]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1)

tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])

In [31]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])*8, dim=-1) # gets too peaky, converges to one-hot

tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])

# 4. Head'i bigram modelinin içine koy ve eğit. Val loss'u 1. görevdeki bigram tabanıyla yan yana koy. Ne kadar düştü, neden düştü, kendi cümlelerinle yaz. Metin üretirken girdiyi neden son block_size harfe kırpmak zorunda olduğunu da yaz.